Veamos si podemos tomar un pequeño modelo y ajustarlo a que emita unicamente sintaxis SQL. Para ello nos apoyaremos en la librería de [transformers](https://huggingface.co/docs/transformers/index) de HuggingFace. Compatible con muchos de los motores de inferencia estándar nos ofrece funcionalidades extra y de abstracción.

![](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/transformers_as_a_model_definition.png)

In [18]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

os.environ["WANDB_DISABLED"] = "true"

In [19]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Modelo
model_name = "facebook/opt-125m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
]
batch = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")

# This is new
batch["labels"] = torch.tensor([1, 1])

optimizer = AdamW(model.parameters())
loss = model(**batch).loss
loss.backward()
optimizer.step()

Some weights of OPTForSequenceClassification were not initialized from the model checkpoint at facebook/opt-125m and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Deberemos crear nuestro conjunto de datos compuesto de:

* La entrada esperada (usuario)
* El contexto añadido (agente)
* Respuesta (llm)

In [20]:
# Function to generate structured prompts for Text-to-SQL tasks
def generate_prompt_sql(input_question, context, output=""):
    return f"""You are a powerful text-to-SQL model. Your job is to answer questions about a database. You are given a question and context regarding one or more tables.
You must output the SQL query that answers the question.
### Input:
{input_question}
### Context:
{context}""", f"""{output}"""

generate_prompt_sql("How many companies do we serve?", context="CREATE TABLE companies (company_id INT)", output="SELECT COUNT(*) FROM companies")

('You are a powerful text-to-SQL model. Your job is to answer questions about a database. You are given a question and context regarding one or more tables. \nYou must output the SQL query that answers the question.\n### Input:\nHow many companies do we serve?\n### Context:\nCREATE TABLE companies (company_id INT)',
 'SELECT COUNT(*) FROM companies')

In [21]:
import pandas as pd

queries = []
responses = []
for item in ["company", "customer","product"]:
    q,r = generate_prompt_sql(f"How many {item} do we serve?", context=f"CREATE TABLE {item} ({item}_id INT)", output=f"SELECT COUNT(*) FROM {item}")
    queries.append(q)
    responses.append(r)

train_data = pd.DataFrame(
    {
        "query" : queries,
        "response" : responses
    }
)
train_data

,query,response
0,You are a powerful text-to-SQL model. Your job...,SELECT COUNT(*) FROM company
1,You are a powerful text-to-SQL model. Your job...,SELECT COUNT(*) FROM customer
2,You are a powerful text-to-SQL model. Your job...,SELECT COUNT(*) FROM product


In [22]:
from datasets import Dataset

# Convert to Hugging Face format
def format_for_hf(examples):
    return {
        "input_ids": tokenizer(examples["query"], truncation=True, padding="max_length", max_length=24)["input_ids"],
        "labels": tokenizer(examples["response"], truncation=True, padding="max_length", max_length=24)["input_ids"]
    }

# Create Hugging Face dataset
hf_dataset = Dataset.from_pandas(train_data)
tokenized_dataset = hf_dataset.map(format_for_hf, batched=True)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [23]:
tokenized_dataset.features

{'query': Value('string'),
 'response': Value('string'),
 'input_ids': List(Value('int32')),
 'labels': List(Value('int64'))}

In [24]:
from transformers import (
    AutoTokenizer,
)

model_name = "facebook/opt-125m"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
max_seq_length = tokenizer.model_max_length

tokenizer("This is the first sentence.", "This is the second one.")

{'input_ids': [2, 713, 16, 5, 78, 3645, 4, 713, 16, 5, 200, 65, 4], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [25]:
tokenizer.convert_ids_to_tokens([2, 713, 16, 5, 78, 3645, 4, 713, 16, 5, 200, 65, 4])

['</s>',
 'This',
 'Ġis',
 'Ġthe',
 'Ġfirst',
 'Ġsentence',
 '.',
 'This',
 'Ġis',
 'Ġthe',
 'Ġsecond',
 'Ġone',
 '.']

In [26]:
from transformers import TrainingArguments

# Configuration for training
training_args = TrainingArguments(
    learning_rate=2e-5,  # Optimal starting point; adjust as needed
    per_device_train_batch_size=4,
    max_steps=200,
    # Additional parameters...
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [27]:
import os
import torch
from transformers import Trainer
from transformers import AutoModelForCausalLM
from transformers import DataCollatorWithPadding

# Para ajustar los tokens del texto
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Modelo a afinar
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=False,
    torch_dtype=torch.float16,
    offload_folder="offload")

# Proceso de entrenamiento
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


Tenéis mucho más contenido en los cursos de HuggingFace: https://huggingface.co/learn/llm-course/chapter3/1?fw=pt